# Sales Intelligence Platform Project

## Import relevant libraries

In [130]:
import numpy as np
import pandas as pd

## Loading the data

In [131]:
FILE_PATH = 'data/raw_data/Sales Dataset - Sales Dataset.csv'

data = pd.read_csv(FILE_PATH)
df = data.copy() #making a copy to preserve the integrity of the original dataset
print(f'Data shape: {df.shape}\n')
df.head()

Data shape: (1194, 12)



,Order ID,Amount,Profit,Quantity,Category,Sub-Category,PaymentMode,Order Date,CustomerName,State,City,Year-Month
0,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2023-06-27,David Padilla,Florida,Miami,2023-06
1,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2024-12
2,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2021-07-25,Robert Stone,New York,Buffalo,2021-07
3,B-26776,4975,1330,14,Electronics,Printers,UPI,2023-06-27,David Padilla,Florida,Miami,2023-06
4,B-26776,4975,1330,14,Electronics,Printers,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2024-12


`Key observations:`
- `Order ID` is not unique. It appears the same for different customers, having different orders, state, city and date.
- `Year-Month` is not useful in the OLTP (repetition). It can be derived from `Order Date` when building the OLAP later.

## Data quality checks and Data cleaning

### Standardising the data columns

In [132]:
df.columns

Index(['Order ID', 'Amount', 'Profit', 'Quantity', 'Category', 'Sub-Category',
       'PaymentMode', 'Order Date', 'CustomerName', 'State', 'City',
       'Year-Month'],
      dtype='object')

In [133]:
import re

df.columns = (
    df.columns
    .str.strip()                                               #remove trailing and leading white spaces
    .map(lambda x: re.sub(r'(?<=[a-z])(?=[A-Z])', '_', x))     #replace camel case in headers with underscore
    .str.replace(r'[\s-]', '_', regex=True)                    #replace white spaces and hyphens in headers with underscore
    .str.lower()                                               #convert to lower case                                             
)

df.columns

Index(['order_id', 'amount', 'profit', 'quantity', 'category', 'sub_category',
       'payment_mode', 'order_date', 'customer_name', 'state', 'city',
       'year_month'],
      dtype='object')

### Dropping the year_month column

In [134]:
df.drop(columns=['year_month'], inplace=True)
df.head()

,order_id,amount,profit,quantity,category,sub_category,payment_mode,order_date,customer_name,state,city
0,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2023-06-27,David Padilla,Florida,Miami
1,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2024-12-27,Connor Morgan,Illinois,Chicago
2,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2021-07-25,Robert Stone,New York,Buffalo
3,B-26776,4975,1330,14,Electronics,Printers,UPI,2023-06-27,David Padilla,Florida,Miami
4,B-26776,4975,1330,14,Electronics,Printers,UPI,2024-12-27,Connor Morgan,Illinois,Chicago


### Duplicated values

In [135]:
# checking for duplicated values
df.duplicated().sum()

0

### Null values

In [136]:
# checking for null values
df.isnull().sum()

order_id         0
amount           0
profit           0
quantity         0
category         0
sub_category     0
payment_mode     0
order_date       0
customer_name    0
state            0
city             0
dtype: int64

### Investigating the Order ID issue earlier observed

In [137]:
total_orders = df['order_id'].nunique() #unique items in the order_id column
problematic_orders = df.groupby('order_id')['customer_name'].nunique()[lambda x: x > 1].count() #customers sharing the same order_id 

print(f'Total unique order IDs: {total_orders}')
print(f'Order IDs shared across multiple customers: {problematic_orders}')
print(f'Percentage problematic: {problematic_orders / total_orders * 100:.2f}%\n') #percentage occurrence in the dataset to measure severity

Total unique order IDs: 547
Order IDs shared across multiple customers: 194
Percentage problematic: 35.47%



It has returned that over a third of order_ids are shared across completely different customers. Therefore, it is unreliable as a unique identifier. This would be considered when designing the ERDs for the OLTP and OLAP. 

### General information on the dataset

In [138]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194 entries, 0 to 1193
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   order_id       1194 non-null   object
 1   amount         1194 non-null   int64 
 2   profit         1194 non-null   int64 
 3   quantity       1194 non-null   int64 
 4   category       1194 non-null   object
 5   sub_category   1194 non-null   object
 6   payment_mode   1194 non-null   object
 7   order_date     1194 non-null   object
 8   customer_name  1194 non-null   object
 9   state          1194 non-null   object
 10  city           1194 non-null   object
dtypes: int64(3), object(8)
memory usage: 102.7+ KB


### Standardising data types for amount, profit and order_date

In [139]:
df = df.astype({'amount': float, 'profit': float})
df['order_date'] = pd.to_datetime(df['order_date'])

In [140]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194 entries, 0 to 1193
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       1194 non-null   object        
 1   amount         1194 non-null   float64       
 2   profit         1194 non-null   float64       
 3   quantity       1194 non-null   int64         
 4   category       1194 non-null   object        
 5   sub_category   1194 non-null   object        
 6   payment_mode   1194 non-null   object        
 7   order_date     1194 non-null   datetime64[ns]
 8   customer_name  1194 non-null   object        
 9   state          1194 non-null   object        
 10  city           1194 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(7)
memory usage: 102.7+ KB


## Defining the OLTP tables

In [141]:
# customers table
customers = df[['customer_name']].copy().drop_duplicates().reset_index(drop=True)
customers.index += 1
customers = customers.reset_index().rename(columns={'index': 'customer_id'})

customers.head()

,customer_id,customer_name
0,1,David Padilla
1,2,Connor Morgan
2,3,Robert Stone
3,4,John Fields
4,5,Clayton Smith


In [142]:
# locations table
locations = df[['city', 'state']].copy().drop_duplicates().reset_index(drop=True)
locations.index += 1
locations = locations.reset_index().rename(columns={'index': 'location_id'})

locations.head()

,location_id,city,state
0,1,Miami,Florida
1,2,Chicago,Illinois
2,3,Buffalo,New York
3,4,Orlando,Florida
4,5,Los Angeles,California


In [143]:
# payment_methods table
payment_methods = df[['payment_mode']].copy().drop_duplicates().reset_index(drop=True)
payment_methods.index += 1
payment_methods = payment_methods.reset_index().rename(columns={'index': 'payment_id'})

payment_methods.head()

,payment_id,payment_mode
0,1,UPI
1,2,Debit Card
2,3,EMI
3,4,Credit Card
4,5,COD


In [165]:
order_items.shape

(1194, 6)

In [144]:
# products table
products = df[['category', 'sub_category']].copy().drop_duplicates().reset_index(drop=True)
products.index += 1
products = products.reset_index().rename(columns={'index': 'product_id'})

products.head()

,product_id,category,sub_category
0,1,Electronics,Electronic Games
1,2,Electronics,Printers
2,3,Office Supplies,Pens
3,4,Electronics,Laptops
4,5,Furniture,Tables


In [145]:
# orders

# Merging all dimension IDs back onto the main dataframe
orders_df = df.merge(customers, on='customer_name') \
              .merge(locations, on=['city', 'state']) \
              .merge(payment_methods, on='payment_mode')

# orders table
orders = orders_df[['customer_id', 'order_id', 'order_date', 'location_id', 'payment_id']] \
                  .copy() \
                  .drop_duplicates() \
                  .reset_index(drop=True)
orders.index += 1
orders = orders.reset_index().rename(columns={'index': 'order_pk'})

orders.head()

,order_pk,customer_id,order_id,order_date,location_id,payment_id
0,1,1,B-26776,2023-06-27,1,1
1,2,110,B-26347,2020-12-21,1,1
2,3,121,B-25400,2021-03-24,1,1
3,4,139,B-25714,2025-02-19,1,1
4,5,258,B-25427,2020-04-04,1,1


In [146]:
# order_items table
order_items = orders_df.merge(orders, on=['order_id', 'customer_id', 'order_date', 'location_id', 'payment_id']) \
                       .merge(products, on=['category', 'sub_category'])

order_items = order_items[['order_pk', 'product_id', 'quantity', 'amount', 'profit']] \
                         .copy() \
                         .reset_index(drop=True)
order_items.index += 1
order_items = order_items.reset_index().rename(columns={'index': 'order_item_id'})

order_items.head()

,order_item_id,order_pk,product_id,quantity,amount,profit
0,1,1,1,5,9726.0,1275.0
1,2,4,1,12,6962.0,3429.0
2,3,6,1,15,3953.0,1776.0
3,4,7,1,20,3409.0,1605.0
4,5,15,1,5,9726.0,1275.0


### Saving the tables to csv

In [147]:
customers.to_csv('data/oltp/customers.csv', index=False)
locations.to_csv('data/oltp/locations.csv', index=False)
products.to_csv('data/oltp/products.csv', index=False)
payment_methods.to_csv('data/oltp/payment_methods.csv', index=False)
orders.to_csv('data/oltp/orders.csv', index=False)
order_items.to_csv('data/oltp/order_items.csv', index=False)

## Defining the OLAP tables

In [148]:
# creating the olap file path
import os
os.makedirs('data/olap', exist_ok=True)

In [149]:
# dim_customer
dim_customer = df[['customer_name']].copy().drop_duplicates().reset_index(drop=True)
dim_customer.index += 1
dim_customer = dim_customer.reset_index().rename(columns={'index': 'customer_id'})

# dim_location
dim_location = df[['city', 'state']].copy().drop_duplicates().reset_index(drop=True)
dim_location.index += 1
dim_location = dim_location.reset_index().rename(columns={'index': 'location_id'})

# dim_product
dim_product = df[['category', 'sub_category']].copy().drop_duplicates().reset_index(drop=True)
dim_product.index += 1
dim_product = dim_product.reset_index().rename(columns={'index': 'product_id'})

# dim_payment
dim_payment = df[['payment_mode']].copy().drop_duplicates().reset_index(drop=True)
dim_payment.index += 1
dim_payment = dim_payment.reset_index().rename(columns={'index': 'payment_id'})

# dim_date
dim_date = df[['order_date']].copy().drop_duplicates().reset_index(drop=True)
dim_date.index += 1
dim_date = dim_date.reset_index().rename(columns={'index': 'date_id'})
dim_date['year_month']  = dim_date['order_date'].dt.to_period('M').astype(str)
dim_date['month']       = dim_date['order_date'].dt.month
dim_date['month_name']  = dim_date['order_date'].dt.strftime('%B')
dim_date['quarter']     = dim_date['order_date'].dt.quarter
dim_date['year']        = dim_date['order_date'].dt.year
dim_date['day_of_week'] = dim_date['order_date'].dt.dayofweek
dim_date['is_weekend']  = dim_date['order_date'].dt.dayofweek >= 5

In [150]:
# fact_order_items
fact_order_items = df.copy() \
    .merge(dim_customer, on='customer_name') \
    .merge(dim_location, on=['city', 'state']) \
    .merge(dim_product, on=['category', 'sub_category']) \
    .merge(dim_payment, on='payment_mode') \
    .merge(dim_date, on='order_date')

fact_order_items = fact_order_items[['customer_id', 'location_id', 'product_id', 'payment_id', 'date_id', 'quantity', 'amount', 'profit']] \
    .copy() \
    .reset_index(drop=True)
fact_order_items.index += 1
fact_order_items = fact_order_items.reset_index().rename(columns={'index': 'order_item_id'})

### Saving the tables to csv

In [151]:
dim_customer.to_csv('data/olap/dim_customer.csv', index=False)
dim_location.to_csv('data/olap/dim_location.csv', index=False)
dim_product.to_csv('data/olap/dim_product.csv', index=False)
dim_payment.to_csv('data/olap/dim_payment.csv', index=False)
dim_date.to_csv('data/olap/dim_date.csv', index=False)
fact_order_items.to_csv('data/olap/fact_order_items.csv', index=False)

## Aggregations and Analysis using the OLAP tables

### Connecting to the OLAP database

In [157]:
import os
from dotenv import load_dotenv

In [158]:
# Loading the .env file
load_dotenv()

DB_HOST     = os.getenv('DB_HOST')
DB_PORT     = os.getenv('DB_PORT')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
OLAP_DB     = os.getenv('OLAP_DB')

# Loading the SQL extension
%load_ext sql

# Connecting to the OLAP database
%sql postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{OLAP_DB}

### Profit and loss by category, city, and customer

In [ ]:
%%sql

SELECT
    dc.customer_name,
    dl.city,
    dl.state,
    dp.category,
    SUM(f.amount)   AS total_revenue,
    SUM(f.profit)   AS total_profit,
    SUM(f.amount) - SUM(f.profit) AS total_loss,
    SUM(f.quantity) AS total_quantity
FROM fact_order_items f
JOIN dim_customer dc ON f.customer_id = dc.customer_id
JOIN dim_location dl ON f.location_id = dl.location_id
JOIN dim_product  dp ON f.product_id  = dp.product_id
GROUP BY
    dc.customer_name,
    dl.city,
    dl.state,
    dp.category
ORDER BY total_profit DESC;

### Order frequency

In [ ]:
%%sql

SELECT
    dc.customer_name,
    COUNT(f.order_item_id) AS order_frequency
FROM fact_order_items f
JOIN dim_customer dc ON f.customer_id = dc.customer_id
GROUP BY dc.customer_name
ORDER BY order_frequency DESC;

### High-profit sub-categories

In [ ]:
%%sql

SELECT
    dp.category,
    dp.sub_category,
    SUM(f.profit) AS total_profit
FROM fact_order_items f
JOIN dim_product dp ON f.product_id = dp.product_id
GROUP BY dp.category, dp.sub_category
HAVING SUM(f.profit) > (
    SELECT AVG(sub_profit)
    FROM (
        SELECT SUM(profit) AS sub_profit
        FROM fact_order_items
        GROUP BY product_id
    ) sub
)
ORDER BY total_profit DESC;

### Peak months

In [ ]:
%%sql

SELECT
    dd.year,
    dd.month,
    dd.month_name,
    SUM(f.profit)          AS total_profit,
    SUM(f.amount)          AS total_revenue,
    COUNT(f.order_item_id) AS total_orders
FROM fact_order_items f
JOIN dim_date dd ON f.date_id = dd.date_id
GROUP BY dd.year, dd.month, dd.month_name
ORDER BY total_profit DESC;

### Cities with revenue above the 95th percentile

In [ ]:
%%sql

WITH city_revenue AS (
    SELECT
        dl.city,
        dl.state,
        SUM(f.amount) AS total_revenue
    FROM fact_order_items f
    JOIN dim_location dl ON f.location_id = dl.location_id
    GROUP BY dl.city, dl.state
),
threshold AS (
    SELECT PERCENTILE_CONT(0.95)
    WITHIN GROUP (ORDER BY total_revenue) AS p95
    FROM city_revenue
)
SELECT
    cr.city,
    cr.state,
    cr.total_revenue,
    t.p95 AS threshold
FROM city_revenue cr
CROSS JOIN threshold t
WHERE cr.total_revenue > t.p95
ORDER BY cr.total_revenue DESC;

### Average monthly sales per product

In [ ]:
%%sql

SELECT
    dp.category,
    dp.sub_category,
    dd.month_name,
    dd.year,
    AVG(f.amount) AS avg_monthly_sales
FROM fact_order_items f
JOIN dim_product dp ON f.product_id = dp.product_id
JOIN dim_date    dd ON f.date_id    = dd.date_id
GROUP BY
    dp.category,
    dp.sub_category,
    dd.month_name,
    dd.year
ORDER BY avg_monthly_sales DESC;

### Aggregated monthly profits, top customers, and quantity sold per category

In [ ]:
%%sql

-- Monthly profits
SELECT
    dd.year,
    dd.month,
    dd.month_name,
    SUM(f.profit) AS monthly_profit
FROM fact_order_items f
JOIN dim_date dd ON f.date_id = dd.date_id
GROUP BY dd.year, dd.month, dd.month_name
ORDER BY dd.year, dd.month;

In [ ]:
%%sql

-- Top customers
SELECT
    dc.customer_name,
    SUM(f.profit)          AS total_profit,
    SUM(f.amount)          AS total_revenue,
    COUNT(f.order_item_id) AS total_orders
FROM fact_order_items f
JOIN dim_customer dc ON f.customer_id = dc.customer_id
GROUP BY dc.customer_name
ORDER BY total_profit DESC
LIMIT 10;

In [ ]:
%%sql

-- Quantity sold per category
SELECT
    dp.category,
    dp.sub_category,
    SUM(f.quantity) AS total_quantity
FROM fact_order_items f
JOIN dim_product dp ON f.product_id = dp.product_id
GROUP BY dp.category, dp.sub_category
ORDER BY total_quantity DESC;

### Filtered cities with low profitability

In [ ]:
%%sql

WITH city_profit AS (
    SELECT
        dl.city,
        dl.state,
        SUM(f.profit) AS total_profit
    FROM fact_order_items f
    JOIN dim_location dl ON f.location_id = dl.location_id
    GROUP BY dl.city, dl.state
),
threshold AS (
    SELECT AVG(total_profit) AS avg_profit
    FROM city_profit
)
SELECT
    cp.city,
    cp.state,
    cp.total_profit,
    t.avg_profit AS benchmark
FROM city_profit cp
CROSS JOIN threshold t
WHERE cp.total_profit < t.avg_profit
ORDER BY cp.total_profit ASC;

In [ ]:
!pip install pymysql
!pip install ipython-sql

!pip install psycopg2
!pip install sqlalchemy

In [ ]:
result = %sql SELECT * FROM fact_order_items LIMIT 5;
df_result = result.DataFrame()
df_result.head()